# Logic-Zero · Colab Quickstart

Run this notebook **first** in a fresh Colab session. It clones the repo, installs deps, sets up secrets, regenerates eval/dev data, then leaves you ready to run the SFT / DPO / GRPO notebooks.

**Recommended runtime:** Runtime → Change runtime type → **L4 GPU** (Colab Pro) or **A100 GPU** (if lucky).

**Before running cell 3 (Secrets):** in Colab left sidebar, click 🔑 Secrets, add:
- `DEEPSEEK_API_KEY` (toggle 'Notebook access')
- (Optional) `HF_TOKEN` for pushing checkpoints in Task 19
- (Optional) `WANDB_API_KEY` for training curve logging

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone repo and install deps

If `pip install -r requirements.txt` fails (spec pins old torch/trl/transformers that may not have wheels for current Colab), we fall back to the relaxed install below.

In [ ]:
# Replace YOUR_USERNAME below with your GitHub username
!git clone https://github.com/YOUR_USERNAME/logic-zero.git
%cd logic-zero

In [ ]:
# Try the pinned versions first
!pip install -q -r requirements.txt 2>&1 | tail -5

In [ ]:
# If above failed: relaxed install (Colab's existing torch + recent trl/peft)
# Uncomment if needed:
# !pip install -q trl==0.13.0 peft==0.13.0 datasets==3.0.0 accelerate==1.0.0 z3-solver wandb python-dotenv openai

In [ ]:
# Verify imports
import torch, transformers, trl, peft, z3, openai
print(f'torch={torch.__version__}  transformers={transformers.__version__}')
print(f'trl={trl.__version__}  peft={peft.__version__}')
print(f'z3={z3.get_version_string()}  openai={openai.__version__}')
print('CUDA available:', torch.cuda.is_available())

## 3. Secrets → .env

Reads from Colab Secrets and writes to `.env`. The file is gitignored.

In [ ]:
from google.colab import userdata
lines = []
for key in ['DEEPSEEK_API_KEY', 'HF_TOKEN', 'WANDB_API_KEY']:
    try:
        val = userdata.get(key)
        lines.append(f'{key}={val}')
        print(f'✓ {key} loaded')
    except Exception:
        print(f'⚠ {key} not set in Colab Secrets (optional except DEEPSEEK_API_KEY)')
open('.env', 'w').write('\n'.join(lines) + '\n')

## 4. Run unit tests

Confirms the data pipeline + parsers + verifier work on Colab. Should be `27 passed, 1 deselected`.

In [ ]:
!pytest train/ data/ -v -m 'not slow' 2>&1 | tail -10

## 5. Generate eval + dev data (~1 minute)

MUST run before any training data generation. Freezes hashes to prevent leak.

In [ ]:
!python -m data.gen_eval_data
!wc -l data/eval_data.jsonl data/dev_data.jsonl
import json
print('Hashes frozen:', len(json.load(open('data/eval_hashes.json'))))

## 6. Persist data to Drive (recommended)

Colab disconnects wipe `/content`. Mounting Drive and copying the generated data avoids re-running. Skip if you don't mind regenerating each session (~1 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/logic-zero/data
!cp data/eval_data.jsonl data/dev_data.jsonl data/eval_hashes.json /content/drive/MyDrive/logic-zero/data/
# Later sessions can restore with:
# !cp /content/drive/MyDrive/logic-zero/data/*.jsonl /content/drive/MyDrive/logic-zero/data/*.json data/

## ✅ You're set up.

**Next steps**, in order:

| Step | Command | Time | Notes |
|---|---|---|---|
| Task 6: SFT data gen | `!python -m data.gen_sft_data --retry-once` | 1-2 h | ~$5 DeepSeek API. Save output to Drive after. |
| Task 7: SFT training | (write `train/sft.py` first per plan §Task 7, then) `!python -m train.sft` | ~2 h on A100 | LoRA, dev-set ckpt selection |
| Task 8: Eval base + SFT | `!python -m eval.run_eval --out results/eval_base.json` then `--adapter results/checkpoints/sft/best --out results/eval_sft.json` | ~3 h | |
| Task 9: Baselines | `!python -m eval.baselines --mode qwen --out results/eval_baseline_qwen.json` + `--mode deepseek` | ~1.5 h | |
| Task 10: Decision gate | Use compare snippet from plan §Task 10 step 1. SFT < 25% → stop, don't go to DPO. | 1 min | |
| Task 11: DPO data | `!python -m data.gen_dpo_data` | 5-10 h | bottleneck = SFT model sampling 4 responses × 2000 puzzles |
| Task 12-20 | Follow plan | | |

**At every checkpoint, push back to GitHub** so you don't lose work to Colab disconnects:
```
!git config user.email "you@example.com"
!git config user.name "Your Name"
!git add results/eval_*.json
!git commit -m "results: SFT eval"
!git push  # requires auth — see below
```

**Pushing from Colab** needs either a HTTPS personal access token in the URL or `gh auth login` with a PAT. The simplest: regenerate the data files (gitignored anyway) and only push model checkpoints to HF Hub (Task 19).